# Multimodal Product Intelligence

### Data Processing

In [76]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [77]:
file_path = 'data/amazon_products.csv'
df_raw = pd.read_csv(file_path)

df = df_raw.rename(columns={ 'title': 'name', 'imgUrl': 'image_url'}).copy()
df.head()

,asin,name,image_url,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",https://m.media-amazon.com/images/I/815dLQKYIY...,https://www.amazon.com/dp/B014TMV5YE,4.5,0.0,139.99,0.00,104.0,False,2000.0
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,https://m.media-amazon.com/images/I/81bQlm7vf6...,https://www.amazon.com/dp/B07GDLCQXV,4.5,0.0,169.99,209.99,104.0,False,1000.0
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,https://m.media-amazon.com/images/I/71EA35zvJB...,https://www.amazon.com/dp/B07XSCCZYG,4.6,0.0,365.49,429.99,104.0,False,300.0
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,https://m.media-amazon.com/images/I/91k6NYLQyI...,https://www.amazon.com/dp/B08MVFKGJM,4.6,0.0,291.59,354.37,104.0,False,400.0
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,https://m.media-amazon.com/images/I/61NJoaZcP9...,https://www.amazon.com/dp/B01DJLKZBA,4.5,0.0,174.99,309.99,104.0,False,400.0


In [78]:
# Drop missing values
df = df.dropna(subset=['name', 'price', 'image_url']).reset_index(drop=True)

# Filter prices
df = df[(df['price'] > 5) & (df['price'] < 500)].reset_index(drop=True)

# 50/50 balance using the median price
median_price = df['price'].median()
df['label'] = (df['price'] > median_price).astype(int)

df_low = df[df['label'] == 0]
df_high = df[df['label'] == 1]

In [79]:
# Sample 1,500 of each class
n = 1500
df_clean_balanced = pd.concat([
    df_low.sample(n, random_state=42),
    df_high.sample(n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Clean, balanced dataset ready: {len(df_clean_balanced)} rows.')
print(f'Median price split: ${median_price:.2f}')

Clean, balanced dataset ready: 3000 rows.
Median price split: $39.99


### Parallel Image Fetching

In [80]:
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

In [81]:
def fetch_image_safe(args):
    idx, url = args
    try:
        # Strict timeout prevents hanging on dead Amazon URLs
        r = requests.get(url, timeout=1.5)
        if r.status_code == 200:
            img = Image.open(BytesIO(r.content)).convert('RGB')
            # Fast resize to save massive local RAM during extraction
            img = img.resize((224, 224))
            return idx, img, True
    except:
        pass
    return idx, None, False


In [82]:
n_tight = 1000
df_low = df_clean_balanced[df_clean_balanced['label'] == 0].sample(n_tight, random_state=42)
df_high = df_clean_balanced[df_clean_balanced['label'] == 1].sample(n_tight, random_state=42)
df_tight = pd.concat([df_low, df_high]).sample(frac=1, random_state=42).reset_index(drop=True)

In [83]:
url_list = list(enumerate(df_tight['image_url'].values))
images   = {}
statuses = {}

print(f'Downloading {len(url_list)} images with strict timeouts...')

In [84]:
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(fetch_image_safe, u): u for u in url_list}
    for i, future in enumerate(as_completed(futures)):
        idx, img, ok = future.result()
        if ok:
            images[idx] = img
        statuses[idx] = ok
    print(f'Success: {sum(statuses.values())}')

Success: 2000


In [85]:
good_idx     = [i for i, ok in statuses.items() if ok]
df_final     = df_tight.iloc[good_idx].reset_index(drop=True)
images_clean = [images[i] for i in good_idx]
prices_clean = df_final['price'].values
labels_clean = df_final['label'].values

print(f'\nFinal active dataset: {len(df_final)}')


Final active dataset: 2000


### Deep Feature Extraction

In [86]:
import torch
from transformers import CLIPProcessor, CLIPModel, DistilBertTokenizer, DistilBertModel
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [87]:
clip_model= CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
bert_tok= DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model= DistilBertModel.from_pretrained('distilbert-base-uncased')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [88]:
# freeze weights
for p in clip_model.parameters(): p.requires_grad = False
for p in bert_model.parameters(): p.requires_grad = False

In [89]:
def extract_embeddings_batched(images, texts, batch_size=16):
    all_img, all_txt = [], []
    n = len(images)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)

        # Image Features
        img_inputs = clip_processor(images=images[start:end], return_tensors='pt', padding=True)
        with torch.no_grad():
            img_emb = clip_model.get_image_features(**img_inputs)
        # Access the pooled output from the BaseModelOutputWithPooling object
        all_img.append(img_emb.pooler_output.cpu().numpy())

        # Text Features
        txt_inputs = bert_tok(texts[start:end], return_tensors='pt', padding=True, truncation=True, max_length=64)
        with torch.no_grad():
            txt_out = bert_model(**txt_inputs)
        all_txt.append(txt_out.last_hidden_state[:, 0, :].cpu().numpy())

    print(f'  Processed batch: {end}/{n}')

    return np.vstack(all_img), np.vstack(all_txt)

In [90]:
print('Starting embedding generation')
img_emb, txt_emb = extract_embeddings_batched(images_clean, df_final['name'].tolist())
fused = np.hstack([img_emb, txt_emb])

os.makedirs('data', exist_ok=True)
np.save('data/fused_emb.npy', fused)
np.save('data/prices.npy',    prices_clean)
print('Success! Embeddings saved cleanly.')

Starting embedding generation
  Processed batch: 2000/2000
Success! Embeddings saved cleanly.


### Train Test split

In [91]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge

X_tr, X_te, y_tr, y_te = train_test_split(fused, prices_clean, test_size=0.2, random_state=42)


In [92]:
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

In [93]:
# Baseline 1: Text-only (DistilBERT -> Ridge)
ridge_txt = Ridge(alpha=1.0).fit(X_tr[:, 512:], y_tr)
smape_txt = smape(y_te, ridge_txt.predict(X_te[:, 512:]))

In [94]:
# Baseline 2: Image-only (CLIP -> Ridge)
ridge_img = Ridge(alpha=1.0).fit(X_tr[:, :512], y_tr)
smape_img = smape(y_te, ridge_img.predict(X_te[:, :512]))

In [95]:
# Baseline 3: Fusion Ridge (No Deep Learning MLP)
ridge_fus = Ridge(alpha=1.0).fit(X_tr, y_tr)
smape_fus_r = smape(y_te, ridge_fus.predict(X_te))

In [96]:
print(f'Text-only Baseline SMAPE: {smape_txt:.2f}%')
print(f'Image-only Baseline SMAPE: {smape_img:.2f}%')
print(f'Fusion Ridge Baseline SMAPE: {smape_fus_r:.2f}%')

Text-only Baseline SMAPE: 69.03%
Image-only Baseline SMAPE: 71.36%
Fusion Ridge Baseline SMAPE: 78.50%


### PyTorch Fusion MLP Training

In [97]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_tr_t = torch.FloatTensor(X_tr)
X_te_t = torch.FloatTensor(X_te)
y_tr_t = torch.FloatTensor(y_tr).unsqueeze(1)


In [98]:
# The Fusion Network (1280 -> 512 -> 128 -> 1)
fusion_mlp = nn.Sequential(
    nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(512,  128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128,    1))

In [99]:
optimizer = torch.optim.Adam( fusion_mlp.parameters(),lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( optimizer, mode='min',factor=0.5, patience=5)
criterion = nn.MSELoss()

dataset = TensorDataset(X_tr_t, y_tr_t)
loader    = DataLoader(dataset, batch_size=64, shuffle=True)

In [100]:
for epoch in range(40):
    fusion_mlp.train()
    epoch_loss = 0
    for bx, by in loader:
        optimizer.zero_grad()
        loss = criterion(fusion_mlp(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_mlp.parameters(),1)
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        fusion_mlp.eval()
        with torch.no_grad():
            val_smape = smape(y_te, fusion_mlp(X_te_t).squeeze().numpy())
        scheduler.step(val_smape)
        print(f'Epoch {epoch+1}/40 | Train MSE Loss: {epoch_loss/len(loader):.4f} | Val SMAPE: {val_smape:.2f}%')

Epoch 10/40 | Train MSE Loss: 1571.3572 | Val SMAPE: 43.32%
Epoch 20/40 | Train MSE Loss: 1230.3952 | Val SMAPE: 44.97%
Epoch 30/40 | Train MSE Loss: 870.8099 | Val SMAPE: 43.65%
Epoch 40/40 | Train MSE Loss: 555.2975 | Val SMAPE: 45.45%


### Evaluation

In [101]:
fusion_mlp.eval()
with torch.no_grad():
    smape_fusion = smape(y_te, fusion_mlp(X_te_t).squeeze().cpu().numpy())


In [106]:
best_baseline = min(smape_txt, smape_img, smape_fus_r)
improvement   = best_baseline - smape_fusion


In [105]:
print(f'\n{"-"*45}')
print(f'{"Model":<35} {"SMAPE":>8}')
print(f'{"-"*45}')
print(f'{"Image-only  (CLIP + Ridge)":<35} {smape_img:>7.2f}%')
print(f'{"Text-only   (DistilBERT + Ridge)":<35} {smape_txt:>7.2f}%')
print(f'{"Fusion      (Ridge, no MLP)":<35} {smape_fus_r:>7.2f}%')
print(f'{"Fusion MLP  (CLIP + DistilBERT)":<35} {smape_fusion:>7.2f}%')
print(f'{"-"*45}')
print(f'Improvement vs best baseline: +{improvement:.2f}% SMAPE')


---------------------------------------------
Model                                  SMAPE
---------------------------------------------
Image-only  (CLIP + Ridge)            71.36%
Text-only   (DistilBERT + Ridge)      69.03%
Fusion      (Ridge, no MLP)           78.50%
Fusion MLP  (CLIP + DistilBERT)       45.45%
---------------------------------------------
Improvement vs best baseline: +23.58% SMAPE


### Conclusion

- Built a multimodal product price prediction system using product images and titles from Amazon listings.
- Extracted visual features using CLIP and textual features using DistilBERT.
- Combined both modalities through a late-fusion MLP architecture to learn price-related patterns.
- Compared the proposed approach against image-only, text-only, and linear fusion baselines.
- The Fusion MLP achieved the best performance with a SMAPE of 45.45%.
- Achieved a 23.58% improvement over the best baseline,showing the effectiveness of multimodal feature fusion.
- Results show that combining visual and textual information leads to more accurate product price prediction than using either modality alone.